# tm_final_10 — Financial Tweet Sentiment: Final Solution

Single end-to-end pipeline using fine-tuned FinBERT (`ProsusAI/finbert`) trained on the full labelled dataset.
Runs top-to-bottom to produce `pred_10.csv`.

In [1]:
from pathlib import Path
import pickle
import sys

import pandas as pd
import torch
from sklearn.model_selection import train_test_split

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "raw"
MODEL_SAVE_DIR = PROJECT_ROOT / "data" / "processed" / "finbert_final_model"
PREDICTIONS_PATH = PROJECT_ROOT / "pred_10.csv"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import models
from preprocessing import ensure_nltk_resource, lowercase_text, regex_clean, stem_text

print("Project root:", PROJECT_ROOT)
print("Model save  :", MODEL_SAVE_DIR)
print("Output      :", PREDICTIONS_PATH)

/Users/mehmet/src/text_mining_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/mehmet/src/text_mining_project
Model save  : /Users/mehmet/src/text_mining_project/data/processed/finbert_final_model
Output      : /Users/mehmet/src/text_mining_project/pred_10.csv


## 1. Load Data

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train: {train.shape}  |  Test: {test.shape}")
train.head(3)

Train: (9543, 2)  |  Test: (2388, 2)


,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0


## 2. Preprocessing

Apply `raw_lower_regex_clean_stemmed` — the best preprocessing variant identified in `tm_tests_10.ipynb`. FinBERT uses the original text directly since its tokenizer expects natural language.

In [3]:
ensure_nltk_resource("corpora/stopwords", "stopwords")

X_all    = train["text"]
y_all    = train["label"]
X_test   = test["text"]
test_ids = test["id"]

# Hold out 10 % for validation reporting — model trains on the remaining 90 %
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.10, stratify=y_all, random_state=73
)

print(f"Train  : {len(X_train)} samples")
print(f"Val    : {len(X_val)} samples")
print(f"Test   : {len(X_test)} samples")

Train  : 8588 samples
Val    : 955 samples
Test   : 2388 samples


## 3. Fine-tune FinBERT

Train `ProsusAI/finbert` for sequence classification. The trained model is saved to `data/processed/finbert_final_model/` on first run and reloaded automatically on every subsequent run — no retraining needed.

In [4]:
FINBERT_MODEL_NAME = "ProsusAI/finbert"
FINBERT_EPOCHS     = 2
FINBERT_BATCH_SIZE = 16
FINBERT_MAX_LENGTH = 128
FINBERT_LR         = 2e-5

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Device: {device}")

Device: mps


In [5]:
_METRICS_CACHE = MODEL_SAVE_DIR / "val_metrics.pkl"

if MODEL_SAVE_DIR.exists() and _METRICS_CACHE.exists():
    print("Saved model found — loading weights, skipping training.")
    with open(_METRICS_CACHE, "rb") as f:
        val_row, val_report = pickle.load(f)
else:
    print("Fine-tuning FinBERT…")
    MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    val_row, val_report = models.train_transformer_classifier(
        variant="finbert_full_finetune_final",
        model_name=FINBERT_MODEL_NAME,
        train_texts=X_train,
        val_texts=X_val,
        y_train=y_train,
        y_val=y_val,
        num_labels=3,
        device=device,
        max_length=FINBERT_MAX_LENGTH,
        batch_size=FINBERT_BATCH_SIZE,
        epochs=FINBERT_EPOCHS,
        learning_rate=FINBERT_LR,
        freeze_encoder=False,
        seed=73,
        save_path=str(MODEL_SAVE_DIR),
    )
    with open(_METRICS_CACHE, "wb") as f:
        pickle.dump((val_row, val_report), f)
    print("Model saved.")

print(f"Macro F1  : {val_row['macro_f1']:.4f}")
print(f"Weighted F1: {val_row['weighted_f1']:.4f}")
print(f"Accuracy  : {val_row['accuracy']:.4f}")

Fine-tuning FinBERT…


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.20it/s]

Model saved.
Macro F1  : 0.8359
Weighted F1: 0.8758
Accuracy  : 0.8743


## 4. Generate Test Predictions

In [6]:
print(f"Running inference on {len(X_test)} test tweets…")

test_preds = models.predict_transformer_classifier(
    model_path=str(MODEL_SAVE_DIR),
    texts=X_test,
    device=device,
    max_length=FINBERT_MAX_LENGTH,
    batch_size=FINBERT_BATCH_SIZE,
)

print(f"Done — {len(test_preds)} predictions")
pd.Series(test_preds).map({0: "Bearish", 1: "Bullish", 2: "Neutral"}).value_counts().to_frame("count")

Running inference on 2388 test tweets…


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19808.63it/s]


Done — 2388 predictions


,count
Neutral,1535
Bullish,459
Bearish,394


## 5. Save `pred_10.csv`

In [7]:
pred_df = pd.DataFrame({"id": test_ids.values, "label": test_preds})
pred_df.to_csv(PREDICTIONS_PATH, index=False)

print(f"Saved {len(pred_df)} rows to: {PREDICTIONS_PATH}")
pred_df.head(10)

Saved 2388 rows to: /Users/mehmet/src/text_mining_project/pred_10.csv


,id,label
0,0,1
1,1,2
2,2,2
3,3,2
4,4,2
5,5,1
6,6,0
7,7,0
8,8,0
9,9,2
